In [47]:
import numpy as np
import math

In [48]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]

encoding = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}

alcohol_content = []
sugar = []
color = []
label = []

for i in data:
    alcohol_content.append(i[0])
    sugar.append(i[1])
    color.append(i[2])
    label.append(encoding.get(i[3]))

X = np.array([alcohol_content,sugar,color]).astype(float).T
y = np.array(label)

In [49]:
def Gini_Impurity(p: list[float]) -> float:
    n = len(p)
    impurity = 1
    for i in range(n):
        impurity -= p[i] * p[i]

    return impurity

In [50]:
def best_split(X, y) -> dict:
    n_samples, n_features = X.shape
    best_gini = float('inf')
    best = {}

    for i in range(n_features):
        thresholds = np.unique(X[:, i])

        for threshold in thresholds:
            left_mask = X[:, i] <= threshold
            right_mask = X[:, i] > threshold

            y_left = y[left_mask]
            y_right = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            n_left = len(y_left)
            count_left = np.bincount(y_left)
            p_left = count_left / n_left

            n_right = len(y_right)
            count_right = np.bincount(y_right)
            p_right = count_right / n_right

            gini_left = Gini_Impurity(p_left)
            gini_right = Gini_Impurity(p_right)

            weighted_gini = (n_left / n_samples) * gini_left + (n_right / n_samples) * gini_right

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best = {'feature_index': i, 'threshold': threshold}

    return best

In [51]:
class Node:
    
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.left = None 
        self.right = None 
        self.value = None 



In [52]:
def build_tree(X, y, depth=0, max_depth=5):
    node = Node()

    if len(set(y)) == 1:
        node.value = y[0]
        return node

    if len(y) < 2 or depth >= max_depth:
        counts = np.bincount(y)
        node.value = np.argmax(counts)
        return node

    best = best_split(X, y)
    if not best:
        counts = np.bincount(y)
        node.value = np.argmax(counts)
        return node

    f = best['feature_index']
    t = best['threshold']

    left_mask = X[:, f] <= t
    right_mask = X[:, f] > t

    X_left, y_left = X[left_mask], y[left_mask]
    X_right, y_right = X[right_mask], y[right_mask]

    node.feature_index = f
    node.threshold = t

    node.left = build_tree(X_left, y_left, depth + 1, max_depth)
    node.right = build_tree(X_right, y_right, depth + 1, max_depth)

    return node

In [53]:
def predict_one(node, x):

    if node.value is not None:
        return node.value

    if x[node.feature_index] <= node.threshold:
        return predict_one(node.left, x)

    else:
        return predict_one(node.right, x)

In [54]:
def predict(node, X):
    
    predictions = []
    for x in X:
        result = predict_one(node, x)
        predictions.append(result)
    
    return predictions

In [55]:

decoding = {0: 'Wine', 1: 'Beer', 2: 'Whiskey'}

tree = build_tree(X, y, max_depth=3)

test_data = np.array([
    [6.0,  2.1,  0],   # Expected: Beer
    [39.0, 0.05, 1],   # Expected: Whiskey
    [13.0, 1.3,  1]    # Expected: Wine
])

predictions = predict(tree, test_data)

for i, pred in enumerate(predictions):
    print(f"Sample {i+1}: {decoding[pred]}")

Sample 1: Wine
Sample 2: Whiskey
Sample 3: Wine
